# RL Project: Atari Tennis Tournament

This notebook implements four Reinforcement Learning algorithms to play Atari Tennis (`ALE/Tennis-v5` via Gymnasium):

1. **SARSA** — Semi-gradient SARSA with linear approximation (inspired by Lab 7, on-policy update from Lab 5B)
2. **Q-Learning** — Off-policy linear approximation (inspired by Lab 5B)
3. **Monte Carlo** — First-visit MC control with linear approximation (inspired by Lab 4)

Each agent is **pre-trained independently** against the built-in Atari AI opponent, then evaluated in a comparative tournament.

In [1]:
import itertools
import pickle
from pathlib import Path

import ale_py  # noqa: F401 — registers ALE environments
import gymnasium as gym
import supersuit as ss
from gymnasium.wrappers import FrameStackObservation, ResizeObservation
from pettingzoo.atari import tennis_v3
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns


In [2]:
CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


def get_path(name: str) -> Path:
    """Return the checkpoint path for an agent (.pkl)."""
    base = name.lower().replace(" ", "_").replace("-", "_")
    return CHECKPOINT_DIR / (base + ".pkl")


# Utility Functions

## Observation Normalization

The Tennis environment produces image observations of shape `(4, 84, 84)` after preprocessing (grayscale + resize + frame stack).
We normalize them into 1D `float64` vectors divided by 255, as in Lab 7 (continuous feature normalization).

## ε-greedy Policy

Follows the pattern from Lab 5B (`epsilon_greedy`) and Lab 7 (`epsilon_greedy_action`):
- With probability ε: random action (exploration)
- With probability 1−ε: action maximizing $\hat{q}(s, a)$ with uniform tie-breaking (`np.flatnonzero`)

In [3]:
def normalize_obs(observation: np.ndarray) -> np.ndarray:
    """Flatten and normalize an observation to a 1D float64 vector.

    Replicates the /255.0 normalization used in all agents from the original project.
    For image observations of shape (4, 84, 84), this produces a vector of length 28_224.

    Args:
        observation: Raw observation array from the environment.

    Returns:
        1D numpy array of dtype float64, values in [0, 1].

    """
    return observation.flatten().astype(np.float64) / 255.0


def epsilon_greedy(
    q_values: np.ndarray,
    epsilon: float,
    rng: np.random.Generator,
) -> int:
    """Select an action using an ε-greedy policy with fair tie-breaking.

    Follows the same logic as Lab 5B epsilon_greedy and Lab 7 epsilon_greedy_action:
    - With probability epsilon: choose a random action (exploration).
    - With probability 1-epsilon: choose the action with highest Q-value (exploitation).
    - If multiple actions share the maximum Q-value, break ties uniformly at random.

    Handles edge cases: empty q_values, NaN/Inf values.

    Args:
        q_values: Array of Q-values for each action, shape (n_actions,).
        epsilon: Exploration probability in [0, 1].
        rng: NumPy random number generator.

    Returns:
        Selected action index.

    """
    q_values = np.asarray(q_values, dtype=np.float64).reshape(-1)

    if q_values.size == 0:
        msg = "q_values is empty."
        raise ValueError(msg)

    if rng.random() < epsilon:
        return int(rng.integers(0, q_values.size))

    # Handle NaN/Inf values safely
    finite_mask = np.isfinite(q_values)
    if not np.any(finite_mask):
        return int(rng.integers(0, q_values.size))

    safe_q = q_values.copy()
    safe_q[~finite_mask] = -np.inf
    max_val = np.max(safe_q)
    best = np.flatnonzero(safe_q == max_val)

    if best.size == 0:
        return int(rng.integers(0, q_values.size))

    return int(rng.choice(best))


# Agent Definitions

## Base Class `Agent`

Common interface for all agents, same signatures: `get_action`, `update`, `save`, `load`.
Serialization uses `pickle` (compatible with numpy arrays).

In [4]:
class Agent:
    """Base class for reinforcement learning agents.

    All agents share this interface so they are compatible with the tournament system.
    """

    def __init__(self, seed: int, action_space: int) -> None:
        """Initialize the agent with its action space and a reproducible RNG."""
        self.action_space = action_space
        self.rng = np.random.default_rng(seed=seed)

    def get_action(self, observation: np.ndarray, epsilon: float = 0.0) -> int:
        """Select an action from the current observation."""
        raise NotImplementedError

    def update(
        self,
        state: np.ndarray,
        action: int,
        reward: float,
        next_state: np.ndarray,
        done: bool,
        next_action: int | None = None,
    ) -> None:
        """Update agent parameters from one transition."""

    def save(self, filename: str) -> None:
        """Save the agent state to disk using pickle."""
        with Path(filename).open("wb") as f:
            pickle.dump(self.__dict__, f)

    def load(self, filename: str) -> None:
        """Load the agent state from disk."""
        with Path(filename).open("rb") as f:
            self.__dict__.update(pickle.load(f))  # noqa: S301


## Random Agent (baseline)

Serves as a reference to evaluate the performance of learning agents.

In [5]:
class RandomAgent(Agent):
    """A simple agent that selects actions uniformly at random (baseline)."""

    def get_action(self, observation: np.ndarray, epsilon: float = 0.0) -> int:
        """Select a random action, ignoring the observation and epsilon."""
        _ = observation, epsilon
        return int(self.rng.integers(0, self.action_space))


## SARSA Agent — Linear Approximation (Semi-gradient)

This agent combines:
- **Linear approximation** from Lab 7 (`SarsaAgent`): $\hat{q}(s, a; \mathbf{W}) = \mathbf{W}_a^\top \phi(s)$
- **On-policy SARSA update** from Lab 5B (`train_sarsa`): $\delta = r + \gamma \hat{q}(s', a') - \hat{q}(s, a)$

The semi-gradient update rule is:
$$W_a \leftarrow W_a + \alpha \cdot \delta \cdot \phi(s)$$

where $\phi(s)$ is the normalized observation vector (analogous to tile coding features in Lab 7, but in dense form).

In [6]:
class SarsaAgent(Agent):
    """Semi-gradient SARSA agent with linear function approximation.

    Inspired by:
    - Lab 7 SarsaAgent: linear q(s,a) = W_a . phi(s), semi-gradient update
    - Lab 5B train_sarsa: on-policy TD target using Q(s', a')

    The weight matrix W has shape (n_actions, n_features).
    For a given state s, q(s, a) = W[a] @ phi(s) is the dot product
    of the action's weight row with the normalized observation.
    """

    def __init__(
        self,
        n_features: int,
        n_actions: int,
        alpha: float = 0.001,
        gamma: float = 0.99,
        seed: int = 42,
    ) -> None:
        """Initialize SARSA agent with linear weights.

        Args:
            n_features: Dimension of the feature vector phi(s).
            n_actions: Number of discrete actions.
            alpha: Learning rate (kept small for high-dim features).
            gamma: Discount factor.
            seed: RNG seed for reproducibility.

        """
        super().__init__(seed, n_actions)
        self.n_features = n_features
        self.alpha = alpha
        self.gamma = gamma
        # Weight matrix: one row per action, analogous to Lab 7's self.w
        # but organized as (n_actions, n_features) for dense features.
        self.W = np.zeros((n_actions, n_features), dtype=np.float64)

    def _q_values(self, phi: np.ndarray) -> np.ndarray:
        """Compute Q-values for all actions given feature vector phi(s).

        Equivalent to Lab 7's self.q(s, a) = self.w[idx].sum()
        but using dense linear approximation: q(s, a) = W[a] @ phi.

        Args:
            phi: Normalized feature vector, shape (n_features,).

        Returns:
            Array of Q-values, shape (n_actions,).

        """
        return self.W @ phi  # shape (n_actions,)

    def get_action(self, observation: np.ndarray, epsilon: float = 0.0) -> int:
        """Select action using ε-greedy policy over linear Q-values.

        Same pattern as Lab 7 SarsaAgent.eps_greedy:
        compute q-values for all actions, then apply epsilon_greedy.
        """
        phi = normalize_obs(observation)
        q_vals = self._q_values(phi)
        return epsilon_greedy(q_vals, epsilon, self.rng)

    def update(
        self,
        state: np.ndarray,
        action: int,
        reward: float,
        next_state: np.ndarray,
        done: bool,
        next_action: int | None = None,
    ) -> None:
        """Perform one semi-gradient SARSA update.

        Follows the SARSA update from Lab 5B train_sarsa:
            td_target = r + gamma * Q(s', a') * (0 if done else 1)
            Q(s, a) += alpha * (td_target - Q(s, a))

        In continuous form with linear approximation (Lab 7 SarsaAgent.update):
            delta = target - q(s, a)
            W[a] += alpha * delta * phi(s)

        Args:
            state: Current observation.
            action: Action taken.
            reward: Reward received.
            next_state: Next observation.
            done: Whether the episode ended.
            next_action: Action chosen in next state (required for SARSA).

        """
        phi = np.nan_to_num(normalize_obs(state), nan=0.0, posinf=0.0, neginf=0.0)
        q_sa = float(self.W[action] @ phi)  # current estimate q(s, a)
        if not np.isfinite(q_sa):
            q_sa = 0.0

        if done:
            # Terminal: no future value (Lab 5B: gamma * Q[s2, a2] * 0)
            target = reward
        else:
            # On-policy: use q(s', a') where a' is the actual next action
            # This is the key SARSA property (Lab 5B)
            phi_next = np.nan_to_num(normalize_obs(next_state), nan=0.0, posinf=0.0, neginf=0.0)
            if next_action is None:
                next_action = 0  # fallback, should not happen in practice
            q_sp_ap = float(self.W[next_action] @ phi_next)
            if not np.isfinite(q_sp_ap):
                q_sp_ap = 0.0
            target = float(reward) + self.gamma * q_sp_ap

        # Semi-gradient update: W[a] += alpha * delta * phi(s)
        # Analogous to Lab 7: self.w[idx] += self.alpha * delta
        if not np.isfinite(target):
            return

        delta = float(target - q_sa)
        if not np.isfinite(delta):
            return

        td_step = float(np.clip(delta, -1_000.0, 1_000.0))
        self.W[action] += self.alpha * td_step * phi
        self.W[action] = np.nan_to_num(self.W[action], nan=0.0, posinf=1e6, neginf=-1e6)


## Q-Learning Agent — Linear Approximation (Off-policy)

Same architecture as SARSA but with the **off-policy update** from Lab 5B (`train_q_learning`):

$$\delta = r + \gamma \max_{a'} \hat{q}(s', a') - \hat{q}(s, a)$$

The key difference from SARSA: we use $\max_{a'} Q(s', a')$ instead of $Q(s', a')$ where $a'$ is the action actually chosen. This allows learning the optimal policy independently of the exploration policy.

In [7]:
class QLearningAgent(Agent):
    """Q-Learning agent with linear function approximation (off-policy).

    Inspired by:
    - Lab 5B train_q_learning: off-policy TD target using max_a' Q(s', a')
    - Lab 7 SarsaAgent: linear approximation q(s,a) = W[a] @ phi(s)

    The only difference from SarsaAgent is the TD target:
    SARSA uses Q(s', a') (on-policy), Q-Learning uses max_a' Q(s', a') (off-policy).
    """

    def __init__(
        self,
        n_features: int,
        n_actions: int,
        alpha: float = 0.001,
        gamma: float = 0.99,
        seed: int = 42,
    ) -> None:
        """Initialize Q-Learning agent with linear weights.

        Args:
            n_features: Dimension of the feature vector phi(s).
            n_actions: Number of discrete actions.
            alpha: Learning rate.
            gamma: Discount factor.
            seed: RNG seed.

        """
        super().__init__(seed, n_actions)
        self.n_features = n_features
        self.alpha = alpha
        self.gamma = gamma
        self.W = np.zeros((n_actions, n_features), dtype=np.float64)

    def _q_values(self, phi: np.ndarray) -> np.ndarray:
        """Compute Q-values for all actions: q(s, a) = W[a] @ phi for each a."""
        return self.W @ phi

    def get_action(self, observation: np.ndarray, epsilon: float = 0.0) -> int:
        """Select action using ε-greedy policy over linear Q-values."""
        phi = normalize_obs(observation)
        q_vals = self._q_values(phi)
        return epsilon_greedy(q_vals, epsilon, self.rng)

    def update(
        self,
        state: np.ndarray,
        action: int,
        reward: float,
        next_state: np.ndarray,
        done: bool,
        next_action: int | None = None,
    ) -> None:
        """Perform one Q-learning update.

        Follows Lab 5B train_q_learning:
            td_target = r + gamma * max(Q[s2]) * (0 if terminated else 1)
            Q[s, a] += alpha * (td_target - Q[s, a])

        In continuous form with linear approximation:
            delta = target - q(s, a)
            W[a] += alpha * delta * phi(s)
        """
        _ = next_action  # Q-learning is off-policy: next_action is not used
        phi = np.nan_to_num(normalize_obs(state), nan=0.0, posinf=0.0, neginf=0.0)
        q_sa = float(self.W[action] @ phi)
        if not np.isfinite(q_sa):
            q_sa = 0.0

        if done:
            # Terminal state: no future value
            # Lab 5B: gamma * np.max(Q[s2]) * (0 if terminated else 1)
            target = reward
        else:
            # Off-policy: use max over all actions in next state
            # This is the key Q-learning property (Lab 5B)
            phi_next = np.nan_to_num(normalize_obs(next_state), nan=0.0, posinf=0.0, neginf=0.0)
            q_next_all = self._q_values(phi_next)  # q(s', a') for all a'
            q_next_max = float(np.max(q_next_all))
            if not np.isfinite(q_next_max):
                q_next_max = 0.0
            target = float(reward) + self.gamma * q_next_max

        if not np.isfinite(target):
            return

        delta = float(target - q_sa)
        if not np.isfinite(delta):
            return

        td_step = float(np.clip(delta, -1_000.0, 1_000.0))
        self.W[action] += self.alpha * td_step * phi
        self.W[action] = np.nan_to_num(self.W[action], nan=0.0, posinf=1e6, neginf=-1e6)


## Monte Carlo Agent — Linear Approximation (First-visit)

This agent is inspired by Lab 4 (`mc_control_epsilon_soft`):
- Accumulates transitions in an episode buffer `(state, action, reward)`
- At the end of the episode (`done=True`), computes **cumulative returns** by traversing the buffer backward:
  $$G \leftarrow \gamma \cdot G + r$$
- Updates weights with the semi-gradient rule:
  $$W_a \leftarrow W_a + \alpha \cdot (G - \hat{q}(s, a)) \cdot \phi(s)$$

Unlike TD methods (SARSA, Q-Learning), Monte Carlo waits for the complete episode to finish before updating.

In [8]:
class MonteCarloAgent(Agent):
    """Monte Carlo control agent with linear function approximation.

    Inspired by Lab 4 mc_control_epsilon_soft:
    - Accumulates transitions in an episode buffer
    - At episode end (done=True), computes discounted returns backward:
        G = gamma * G + r  (same as Lab 4's reversed loop)
    - Updates weights with semi-gradient: W[a] += alpha * (G - q(s,a)) * phi(s)

    Unlike TD methods (SARSA, Q-Learning), no update occurs until the episode ends.

    Performance optimizations over naive per-step implementation:
    - float32 weights & features (halves memory bandwidth, faster SIMD)
    - Raw observations stored compactly as uint8, batch-normalized at episode end
    - Vectorized return computation & chunk-based weight updates via einsum
    - Single weight sanitization per episode instead of per-step
    """

    def __init__(
        self,
        n_features: int,
        n_actions: int,
        alpha: float = 0.001,
        gamma: float = 0.99,
        seed: int = 42,
    ) -> None:
        super().__init__(seed, n_actions)
        self.n_features = n_features
        self.alpha = alpha
        self.gamma = gamma
        self.W = np.zeros((n_actions, n_features), dtype=np.float32)
        self._obs_buf: list[np.ndarray] = []
        self._act_buf: list[int] = []
        self._rew_buf: list[float] = []

    def _q_values(self, phi: np.ndarray) -> np.ndarray:
        return self.W @ phi

    def get_action(self, observation: np.ndarray, epsilon: float = 0.0) -> int:
        phi = observation.flatten().astype(np.float32) / np.float32(255.0)
        q_vals = self._q_values(phi)
        return epsilon_greedy(q_vals, epsilon, self.rng)

    def update(
        self,
        state: np.ndarray,
        action: int,
        reward: float,
        next_state: np.ndarray,
        done: bool,
        next_action: int | None = None,
    ) -> None:
        _ = next_state, next_action

        self._obs_buf.append(state)
        self._act_buf.append(action)
        self._rew_buf.append(reward)

        if not done:
            return

        n = len(self._rew_buf)
        actions = np.array(self._act_buf, dtype=np.intp)

        returns = np.empty(n, dtype=np.float32)
        G = np.float32(0.0)
        gamma32 = np.float32(self.gamma)
        for i in range(n - 1, -1, -1):
            G = gamma32 * G + np.float32(self._rew_buf[i])
            returns[i] = G

        alpha32 = np.float32(self.alpha)
        chunk_size = 500
        for start in range(0, n, chunk_size):
            end = min(start + chunk_size, n)
            cs = end - start

            raw = np.array(self._obs_buf[start:end])
            phi = raw.reshape(cs, -1).astype(np.float32)
            phi /= np.float32(255.0)

            ca = actions[start:end]
            q_sa = np.einsum("ij,ij->i", self.W[ca], phi)

            deltas = np.clip(returns[start:end] - q_sa, -1000.0, 1000.0)

            for a in range(self.action_space):
                mask = ca == a
                if not np.any(mask):
                    continue
                self.W[a] += alpha32 * (deltas[mask] @ phi[mask])

        self.W = np.nan_to_num(self.W, nan=0.0, posinf=1e6, neginf=-1e6)

        self._obs_buf.clear()
        self._act_buf.clear()
        self._rew_buf.clear()


## Tennis Environment

Creation of the Atari Tennis environment via Gymnasium (`ALE/Tennis-v5`) with standard wrappers:
- **Grayscale**: `obs_type="grayscale"` — single-channel observations
- **Resize**: `ResizeObservation(84, 84)` — downscale to 84×84
- **Frame stack**: `FrameStackObservation(4)` — stack 4 consecutive frames

The final observation is an array of shape `(4, 84, 84)`, which flattens to 28,224 features.

The agent plays against the **built-in Atari AI opponent**.

In [9]:
def create_env() -> gym.Env:
    """Create the ALE/Tennis-v5 environment with preprocessing wrappers.

    Applies:
    - obs_type="grayscale": grayscale observation (210, 160)
    - ResizeObservation(84, 84): downscale to 84x84
    - FrameStackObservation(4): stack 4 consecutive frames -> (4, 84, 84)

    Returns:
        Gymnasium environment ready for training.

    """
    env = gym.make("ALE/Tennis-v5", obs_type="grayscale")
    env = ResizeObservation(env, shape=(84, 84))
    return FrameStackObservation(env, stack_size=4)


## Training & Evaluation Infrastructure

Functions for training and evaluating agents in the single-agent Gymnasium environment:

1. **`train_agent`** — Pre-trains an agent against the built-in AI for a given number of episodes with ε-greedy exploration
2. **`evaluate_agent`** — Evaluates a trained agent (no exploration, ε = 0) and returns performance metrics
3. **`plot_training_curves`** — Plots the training reward history (moving average) for all agents
4. **`plot_evaluation_comparison`** — Bar chart comparing final evaluation scores across agents
5. **`evaluate_tournament`** — Evaluates all agents and produces a summary comparison

In [10]:
def train_agent(
    env: gym.Env,
    agent: Agent,
    name: str,
    *,
    episodes: int = 5000,
    epsilon_start: float = 1.0,
    epsilon_end: float = 0.05,
    epsilon_decay: float = 0.999,
    max_steps: int = 5000,
) -> list[float]:
    """Pre-train an agent against the built-in Atari AI opponent.

    Each agent learns independently by playing full episodes. This is the
    self-play pre-training phase: the agent interacts with the environment's
    built-in opponent and updates its parameters after each transition.

    Args:
        env: Gymnasium ALE/Tennis-v5 environment.
        agent: Agent instance to train.
        name: Display name for the progress bar.
        episodes: Number of training episodes.
        epsilon_start: Initial exploration rate.
        epsilon_end: Minimum exploration rate.
        epsilon_decay: Multiplicative decay per episode.
        max_steps: Maximum steps per episode.

    Returns:
        List of total rewards per episode.

    """
    rewards_history: list[float] = []
    epsilon = epsilon_start

    pbar = tqdm(range(episodes), desc=f"Training {name}", leave=True)

    for _ep in pbar:
        obs, _info = env.reset()
        obs = np.asarray(obs)
        total_reward = 0.0

        # Select first action
        action = agent.get_action(obs, epsilon=epsilon)

        for _step in range(max_steps):
            next_obs, reward, terminated, truncated, _info = env.step(action)
            next_obs = np.asarray(next_obs)
            done = terminated or truncated
            reward = float(reward)
            total_reward += reward

            # Select next action (needed for SARSA's on-policy update)
            next_action = agent.get_action(next_obs, epsilon=epsilon) if not done else None

            # Update agent with the transition
            agent.update(
                state=obs,
                action=action,
                reward=reward,
                next_state=next_obs,
                done=done,
                next_action=next_action,
            )

            if done:
                break

            obs = next_obs
            action = next_action

        rewards_history.append(total_reward)
        epsilon = max(epsilon_end, epsilon * epsilon_decay)

        # Update progress bar
        recent_window = 50
        if len(rewards_history) >= recent_window:
            recent_avg = np.mean(rewards_history[-recent_window:])
            pbar.set_postfix(
            avg50=f"{recent_avg:.1f}",
            eps=f"{epsilon:.3f}",
            rew=f"{total_reward:.0f}",
            )

    return rewards_history


def evaluate_agent(
    env: gym.Env,
    agent: Agent,
    name: str,
    *,
    episodes: int = 20,
    max_steps: int = 5000,
) -> dict[str, object]:
    """Evaluate a trained agent with no exploration (ε = 0).

    Args:
        env: Gymnasium ALE/Tennis-v5 environment.
        agent: Trained agent to evaluate.
        name: Display name for the progress bar.
        episodes: Number of evaluation episodes.
        max_steps: Maximum steps per episode.

    Returns:
        Dictionary with rewards list, mean, std, wins, and win rate.

    """
    rewards: list[float] = []
    wins = 0

    for _ep in tqdm(range(episodes), desc=f"Evaluating {name}", leave=False):
        obs, _info = env.reset()
        total_reward = 0.0

        for _step in range(max_steps):
            action = agent.get_action(np.asarray(obs), epsilon=0.0)
            obs, reward, terminated, truncated, _info = env.step(action)
            reward = float(reward)
            total_reward += reward
            if terminated or truncated:
                break

        rewards.append(total_reward)
        if total_reward > 0:
            wins += 1

    return {
        "rewards": rewards,
        "mean_reward": float(np.mean(rewards)),
        "std_reward": float(np.std(rewards)),
        "wins": wins,
        "win_rate": wins / episodes,
    }


def plot_training_curves(
    training_histories: dict[str, list[float]],
    path: str,
    window: int = 100,
) -> None:
    """Plot training reward curves for all agents on a single figure.

    Uses a moving average to smooth the curves.

    Args:
        training_histories: Dict mapping agent names to reward lists.
        path: File path to save the plot image.
        window: Moving average window size.

    """
    plt.figure(figsize=(12, 6))

    for name, rewards in training_histories.items():
        if len(rewards) >= window:
            ma = np.convolve(rewards, np.ones(window) / window, mode="valid")
            plt.plot(np.arange(window - 1, len(rewards)), ma, label=name)
        else:
            plt.plot(rewards, label=f"{name} (raw)")

    plt.xlabel("Episodes")
    plt.ylabel(f"Average Reward (Window={window})")
    plt.title("Training Curves (vs built-in AI)")
    plt.legend()
    plt.grid(visible=True)
    plt.tight_layout()
    plt.savefig(path)
    plt.show()


def plot_evaluation_comparison(results: dict[str, dict[str, object]]) -> None:
    """Bar chart comparing evaluation performance of all agents.

    Args:
        results: Dict mapping agent names to evaluation result dicts.

    """
    names = list(results.keys())
    means = [results[n]["mean_reward"] for n in names]
    stds = [results[n]["std_reward"] for n in names]
    win_rates = [results[n]["win_rate"] for n in names]

    _fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Mean reward bar chart
    colors = sns.color_palette("husl", len(names))
    axes[0].bar(names, means, yerr=stds, capsize=5, color=colors, edgecolor="black")
    axes[0].set_ylabel("Mean Reward")
    axes[0].set_title("Evaluation: Mean Reward per Agent (vs built-in AI)")
    axes[0].axhline(y=0, color="gray", linestyle="--", alpha=0.5)
    axes[0].grid(axis="y", alpha=0.3)

    # Win rate bar chart
    axes[1].bar(names, win_rates, color=colors, edgecolor="black")
    axes[1].set_ylabel("Win Rate")
    axes[1].set_title("Evaluation: Win Rate per Agent (vs built-in AI)")
    axes[1].set_ylim(0, 1)
    axes[1].axhline(y=0.5, color="gray", linestyle="--", alpha=0.5, label="50% baseline")
    axes[1].legend()
    axes[1].grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()


def evaluate_tournament(
    env: gym.Env,
    agents: dict[str, Agent],
    episodes_per_agent: int = 20,
) -> dict[str, dict[str, object]]:
    """Evaluate all agents against the built-in AI and produce a comparison.

    Args:
        env: Gymnasium ALE/Tennis-v5 environment.
        agents: Dictionary mapping agent names to Agent instances.
        episodes_per_agent: Number of evaluation episodes per agent.

    Returns:
        Dict mapping agent names to their evaluation results.

    """
    results: dict[str, dict[str, object]] = {}
    n_agents = len(agents)

    for idx, (name, agent) in enumerate(agents.items(), start=1):
        print(f"[Evaluation {idx}/{n_agents}] {name}")
        results[name] = evaluate_agent(
            env, agent, name, episodes=episodes_per_agent,
        )
        mean_r = results[name]["mean_reward"]
        wr = results[name]["win_rate"]
        print(f"  -> Mean reward: {mean_r:.2f} | Win rate: {wr:.1%}\n")

    return results


## Agent Instantiation & Incremental Training (One Agent at a Time)

**Environment**: `ALE/Tennis-v5` (grayscale, 84×84×4 frames → 28,224 features, 18 actions).

**Agents**:
- **Random** — random baseline (no training needed)
- **SARSA** — linear approximation, semi-gradient TD(0)
- **Q-Learning** — linear approximation, off-policy
- **Monte Carlo** — linear approximation, first-visit returns

**Workflow**:
1. Train **one** selected agent (`AGENT_TO_TRAIN`)
2. Save its weights to `checkpoints/` (`.pkl`)
3. Repeat later for another agent without retraining previous ones
4. Load all saved checkpoints before the final evaluation

In [11]:
# Create environment
env = create_env()
obs, _info = env.reset()

n_actions = int(env.action_space.n)
n_features = int(np.prod(obs.shape))

print(f"Observation shape : {obs.shape}")
print(f"Feature vector dim: {n_features}")
print(f"Number of actions : {n_actions}")

# Instantiate agents
agent_random = RandomAgent(seed=42, action_space=int(n_actions))
agent_sarsa = SarsaAgent(n_features=n_features, n_actions=n_actions, alpha=1e-5)
agent_q = QLearningAgent(n_features=n_features, n_actions=n_actions, alpha=1e-5)
agent_mc = MonteCarloAgent(n_features=n_features, n_actions=n_actions, alpha=1e-5)

agents = {
    "Random": agent_random,
    "SARSA": agent_sarsa,
    "Q-Learning": agent_q,
    "Monte Carlo": agent_mc,
}


A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]


Observation shape : (4, 84, 84)
Feature vector dim: 28224
Number of actions : 18


objc[49878]: Class SDLApplication is implemented in both /Users/arthurdanjou/Workspace/studies/.venv/lib/python3.13/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x11118d2c8) and /Users/arthurdanjou/Workspace/studies/.venv/lib/python3.13/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x124418890). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[49878]: Class SDLAppDelegate is implemented in both /Users/arthurdanjou/Workspace/studies/.venv/lib/python3.13/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x11118d318) and /Users/arthurdanjou/Workspace/studies/.venv/lib/python3.13/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x1244188e0). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[49878]: Class SDLTranslatorResponder is implemented in both /Users/arthurdanjou/Workspace/studies/.venv/lib/python3.13/site-packages/pygame/.dylibs/libSDL2-2.0

In [ ]:
AGENT_TO_TRAIN = "Monte Carlo"  # TODO: change to: "Q-Learning", "Monte Carlo", "Random"
TRAINING_EPISODES = 2500
FORCE_RETRAIN = False

if AGENT_TO_TRAIN not in agents:
    msg = f"Unknown agent '{AGENT_TO_TRAIN}'. Available: {list(agents)}"
    raise ValueError(msg)

training_histories: dict[str, list[float]] = {}
agent = agents[AGENT_TO_TRAIN]
checkpoint_path = get_path(AGENT_TO_TRAIN)

print(f"Selected agent: {AGENT_TO_TRAIN}")
print(f"Checkpoint path: {checkpoint_path}")

if AGENT_TO_TRAIN == "Random":
    print("Random is a baseline and is not trained.")
    training_histories[AGENT_TO_TRAIN] = []
elif checkpoint_path.exists() and not FORCE_RETRAIN:
    agent.load(str(checkpoint_path))
    print("Checkpoint found -> weights loaded, training skipped.")
    training_histories[AGENT_TO_TRAIN] = []
else:
    print(f"\n{'='*60}")
    print(f"Training: {AGENT_TO_TRAIN} ({TRAINING_EPISODES} episodes)")
    print(f"{'='*60}")

    training_histories[AGENT_TO_TRAIN] = train_agent(
        env=env,
        agent=agent,
        name=AGENT_TO_TRAIN,
        episodes=TRAINING_EPISODES,
        epsilon_start=1.0,
        epsilon_end=0.05,
        epsilon_decay=0.999,
    )

    avg_last_100 = np.mean(training_histories[AGENT_TO_TRAIN][-100:])
    print(f"-> {AGENT_TO_TRAIN} avg reward (last 100 eps): {avg_last_100:.2f}")

    agent.save(str(checkpoint_path))
    print("Checkpoint saved.")


Selected agent: Monte Carlo
Checkpoint path: checkpoints/monte_carlo.pkl

Training: Monte Carlo (2500 episodes)


Training Monte Carlo:   0%|          | 0/2500 [00:00<?, ?it/s]

In [ ]:
plot_training_curves(
    training_histories, f"plots/{AGENT_TO_TRAIN}_training_curves.png", window=100,
)


## Final Evaluation

Each agent plays 20 episodes against the built-in AI with no exploration (ε = 0).
Performance is compared via mean reward and win rate.

In [ ]:
# Build evaluation set: Random + agents with existing checkpoints
eval_agents: dict[str, Agent] = {"Random": agents["Random"]}
missing_agents: list[str] = []

for name, agent in agents.items():
    if name == "Random":
        continue

    checkpoint_path = get_path(name)

    if checkpoint_path.exists():
        agent.load(str(checkpoint_path))
        eval_agents[name] = agent
    else:
        missing_agents.append(name)

print(f"Agents evaluated: {list(eval_agents.keys())}")
if missing_agents:
    print(f"Skipped (no checkpoint yet): {missing_agents}")

if len(eval_agents) < 2:
    raise RuntimeError("Train at least one non-random agent before final evaluation.")

results = evaluate_tournament(env, eval_agents, episodes_per_agent=20)
plot_evaluation_comparison(results)

# Print summary table
print(f"\n{'Agent':<15} {'Mean Reward':>12} {'Std':>8} {'Win Rate':>10}")
print("-" * 48)
for name, res in results.items():
    print(f"{name:<15} {res['mean_reward']:>12.2f} {res['std_reward']:>8.2f} {res['win_rate']:>9.1%}")


In [ ]:
def create_tournament_env():
    """Create PettingZoo Tennis env with preprocessing compatible with our agents."""
    env = tennis_v3.env(obs_type="rgb_image")
    env = ss.color_reduction_v0(env, mode="full")
    env = ss.resize_v1(env, x_size=84, y_size=84)
    return ss.frame_stack_v1(env, 4)


def run_pz_match(
    env,
    agent_first: Agent,
    agent_second: Agent,
    episodes: int = 10,
    max_steps: int = 4000,
) -> dict[str, int]:
    """Run multiple PettingZoo episodes between two agents.

    Returns wins for global labels {'first': ..., 'second': ..., 'draw': ...}.
    """
    wins = {"first": 0, "second": 0, "draw": 0}

    for _ep in range(episodes):
        env.reset()
        rewards = {"first_0": 0.0, "second_0": 0.0}

        for step_idx, agent_id in enumerate(env.agent_iter()):
            obs, reward, termination, truncation, _info = env.last()
            done = termination or truncation
            rewards[agent_id] += float(reward)

            if done or step_idx >= max_steps:
                action = None
            else:
                current_agent = agent_first if agent_id == "first_0" else agent_second
                action = current_agent.get_action(np.asarray(obs), epsilon=0.0)

            env.step(action)

            if step_idx + 1 >= max_steps:
                break

        if rewards["first_0"] > rewards["second_0"]:
            wins["first"] += 1
        elif rewards["second_0"] > rewards["first_0"]:
            wins["second"] += 1
        else:
            wins["draw"] += 1

    return wins


def run_pettingzoo_tournament(
    agents: dict[str, Agent],
    episodes_per_side: int = 10,
) -> tuple[np.ndarray, list[str]]:
    """Round-robin tournament excluding Random, with seat-swap fairness."""
    _ = itertools  # kept for notebook context consistency
    candidate_names = [name for name in agents if name != "Random"]

    # Keep only agents that have a checkpoint
    ready_names: list[str] = []
    for name in candidate_names:
        checkpoint_path = get_path(name)
        if checkpoint_path.exists():
            agents[name].load(str(checkpoint_path))
            ready_names.append(name)

    if len(ready_names) < 2:
        msg = "Need at least 2 trained (checkpointed) non-random agents for PettingZoo tournament."
        raise RuntimeError(msg)

    n = len(ready_names)
    win_matrix = np.full((n, n), np.nan)
    np.fill_diagonal(win_matrix, 0.5)

    for i in range(n):
        for j in range(i + 1, n):
            name_i = ready_names[i]
            name_j = ready_names[j]

            print(f"Matchup: {name_i} vs {name_j}")
            env = create_tournament_env()

            # Leg 1: i as first_0, j as second_0
            leg1 = run_pz_match(
                env,
                agent_first=agents[name_i],
                agent_second=agents[name_j],
                episodes=episodes_per_side,
            )

            # Leg 2: swap seats
            leg2 = run_pz_match(
                env,
                agent_first=agents[name_j],
                agent_second=agents[name_i],
                episodes=episodes_per_side,
            )

            env.close()

            wins_i = leg1["first"] + leg2["second"]
            wins_j = leg1["second"] + leg2["first"]

            decisive = wins_i + wins_j
            if decisive == 0:
                wr_i = 0.5
                wr_j = 0.5
            else:
                wr_i = wins_i / decisive
                wr_j = wins_j / decisive

            win_matrix[i, j] = wr_i
            win_matrix[j, i] = wr_j

            print(f"  -> {name_i}: {wins_i} wins | {name_j}: {wins_j} wins\n")

    return win_matrix, ready_names


# Run tournament (non-random agents only)
win_matrix_pz, pz_names = run_pettingzoo_tournament(
    agents=agents,
    episodes_per_side=10,
)

# Plot win-rate matrix
plt.figure(figsize=(8, 6))
sns.heatmap(
    win_matrix_pz,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    vmin=0.0,
    vmax=1.0,
    xticklabels=pz_names,
    yticklabels=pz_names,
)
plt.xlabel("Opponent")
plt.ylabel("Agent")
plt.title("PettingZoo Tournament Win Rate Matrix (Non-random agents)")
plt.tight_layout()
plt.show()

# Rank agents by mean win rate vs others (excluding diagonal)
scores = {}
for idx, name in enumerate(pz_names):
    row = np.delete(win_matrix_pz[idx], idx)
    scores[name] = float(np.mean(row))

ranking = sorted(scores.items(), key=lambda x: x[1], reverse=True)
print("Final ranking (PettingZoo tournament, non-random):")
for rank_idx, (name, score) in enumerate(ranking, start=1):
    print(f"{rank_idx}. {name:<12} | mean win rate: {score:.3f}")

print(f"\nBest agent: {ranking[0][0]}")


## PettingZoo Tournament (Agents vs Agents)

This tournament uses `from pettingzoo.atari import tennis_v3` to make trained agents play against each other directly.

- Checkpoints are loaded from `checkpoints/` (`.pkl`)
- `Random` is **excluded** from ranking
- Each pair plays in both seat positions (`first_0` and `second_0`) to reduce position bias
- A win-rate matrix and final ranking are produced